# Hugging Face raster ZIPs to model-ready NumPy arrays

This Kaggle notebook converts the three labeled source folders into mmap-friendly structured
`.npy` files, leakage-safe incident-group splits, manifests, train-only statistics, and
per-source test arrays. It processes one archive at a time and checkpoints sample progress.

**Run a smoke test first:** set `DEBUG_MAX_SAMPLES` to a small value (for example 30), run
through verification, then switch it to `None` and use a fresh `WORK_DIR` for the full run.
This environment has no Kaggle secrets or source-repository access, so the remote pipeline
must be run in Kaggle after filling in the configuration.


In [ ]:
# Configuration
import numpy as np

SOURCE_REPO = "<user>/<source-dataset>"
DEST_REPO = "<user>/<destination-dataset>"

SOURCE_LAYOUT = {
    "landslide_early": {"label": 1, "source_id": 0},
    "landslide_latest": {"label": 1, "source_id": 1},
    "non_landslide": {"label": 0, "source_id": 2},
}

SEED = 42
SPLIT = {"train": 0.70, "test": 0.20, "val": 0.10}
OUTPUT_DTYPE = "float32"  # choose "float16" to reduce storage
WORK_DIR = "/kaggle/working"
DEBUG_MAX_SAMPLES = None  # set to e.g. 30 for a smoke test before the full run
S1_INPUT_SCALE = "auto"  # "auto", "db", "linear_power", or "linear_amplitude"
S1_CLIP_PERCENTILES = (1.0, 99.0)
SHAPE_RESOLUTION_TOLERANCE = 0.02
S1_CALIBRATION_SAMPLES = 32

if SOURCE_REPO.startswith("<") or DEST_REPO.startswith("<"):
    raise ValueError("Fill in SOURCE_REPO and DEST_REPO before running this notebook.")
if SOURCE_REPO == DEST_REPO:
    raise ValueError("SOURCE_REPO and DEST_REPO must be different datasets.")
if OUTPUT_DTYPE not in {"float32", "float16"}:
    raise ValueError("OUTPUT_DTYPE must be 'float32' or 'float16'.")
if S1_INPUT_SCALE not in {"auto", "db", "linear_power", "linear_amplitude"}:
    raise ValueError("Invalid S1_INPUT_SCALE.")
if not np.isclose(sum(SPLIT.values()), 1.0) or any(value <= 0 for value in SPLIT.values()):
    raise ValueError("SPLIT fractions must sum to 1.")
if not (0 <= S1_CLIP_PERCENTILES[0] < S1_CLIP_PERCENTILES[1] <= 100):
    raise ValueError("S1_CLIP_PERCENTILES must be increasing values between 0 and 100.")


In [ ]:
# Install dependencies and authenticate without displaying the token
%pip -q install huggingface_hub rasterio pandas pyarrow scikit-learn tqdm matplotlib

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        'Add a Kaggle secret named "HF_TOKEN" with Hugging Face write access.'
    ) from exc
if not HF_TOKEN:
    raise RuntimeError('Kaggle secret "HF_TOKEN" is empty.')

api = HfApi(token=HF_TOKEN)
try:
    SOURCE_REVISION = api.repo_info(repo_id=SOURCE_REPO, repo_type="dataset").sha
except Exception as exc:
    raise RuntimeError(
        f"Cannot access source dataset {SOURCE_REPO!r}; check the repo ID and HF_TOKEN read access."
    ) from exc
print(f"Authenticated and verified the source dataset at revision {SOURCE_REVISION}.")

import hashlib
import json
import logging
import math
import os
import random
import re
import shutil
import tempfile
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform, transform_bounds
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOGGER = logging.getLogger("hf_numpy_export")
WORK = Path(WORK_DIR)
WORK.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR = WORK / "_downloads"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
LAYERS = {
    "planet_before": "planet_before",
    "planet_after": "planet_after",
    "s1_before": "s1_before",
    "s1_after": "s1_after",
    "slope": "slope",
    "aspect": "aspect",
}
ID_PATTERN = re.compile(r"^\d+_\d+$")


## 1. Discover ZIPs and inspect the source

The index is derived from ZIP member names, with only raster headers read during this phase.
Archives are downloaded one at a time and removed after inspection. CSVs may be in an archive
or next to it in the source folder.


In [ ]:
repo_files = sorted(api.list_repo_files(
    repo_id=SOURCE_REPO, repo_type="dataset", revision=SOURCE_REVISION
))
archives = []
for source_folder in SOURCE_LAYOUT:
    found = [p for p in repo_files if p.lower().endswith(".zip")
             and PurePosixPath(p).parent.name == source_folder]
    if not found:
        raise FileNotFoundError(f"No .zip files found in source folder {source_folder!r}.")
    archives.extend({"path": p, "source_folder": source_folder} for p in found)

print("ZIP files found per source folder:")
for source_folder in SOURCE_LAYOUT:
    names = [a["path"] for a in archives if a["source_folder"] == source_folder]
    print(f"  {source_folder}: {len(names)}")
    for name in names:
        print(f"    {name}")

INDEX_FILE = WORK / "sample_index.json"
archive_signature = hashlib.sha256(
    json.dumps([SOURCE_REPO, SOURCE_REVISION, repo_files, SOURCE_LAYOUT], sort_keys=True).encode()
).hexdigest()

def fetch_zip(repo_path):
    """Download one archive into WORK_DIR; callers must unlink it in a finally block."""
    return Path(hf_hub_download(
        repo_id=SOURCE_REPO, filename=repo_path, repo_type="dataset",
        token=HF_TOKEN, revision=SOURCE_REVISION, local_dir=str(DOWNLOAD_DIR), force_download=True,
    ))

def member_csv(zf, archive, kind):
    """Return one archive's CSV DataFrame, falling back to a neighboring repo CSV."""
    names = zf.namelist()
    candidates = [n for n in names if PurePosixPath(n).name.lower() == f"{kind}.csv"]
    stem = Path(archive["path"]).stem.lower()
    candidates += [n for n in names if PurePosixPath(n).name.lower()
                   in {f"{stem}_{kind}.csv", f"{stem}-{kind}.csv"}]
    if candidates:
        member = sorted(set(candidates))[0]
        with zf.open(member) as stream:
            return pd.read_csv(stream), f"zip:{member}"
    folder = archive["source_folder"]
    external = [p for p in repo_files if PurePosixPath(p).parent.name == folder
                and PurePosixPath(p).name.lower() in {f"{kind}.csv",
                    f"{stem}_{kind}.csv", f"{stem}-{kind}.csv"}]
    if external:
        remote_path = sorted(external)[0]
        local_path = hf_hub_download(
            repo_id=SOURCE_REPO, filename=remote_path, repo_type="dataset",
            token=HF_TOKEN, revision=SOURCE_REVISION,
        )
        return pd.read_csv(local_path), remote_path
    return pd.DataFrame(), None

def get_raster_profile(zip_path, member):
    """Read raster metadata via GDAL's ZIP virtual filesystem without extracting pixels."""
    uri = f"zip://{zip_path}!/{member}"
    with rasterio.open(uri) as ds:
        return {
            "count": ds.count, "height": ds.height, "width": ds.width,
            "dtypes": list(ds.dtypes), "crs": ds.crs.to_string() if ds.crs else None,
            "transform": list(ds.transform), "bounds": list(ds.bounds),
            "res": list(ds.res), "nodata": ds.nodata,
            "descriptions": list(ds.descriptions),
        }

def discover_archive(archive):
    """Build the cheap per-sample index and inspect TIFF headers in one ZIP."""
    zip_path = fetch_zip(archive["path"])
    try:
        with zipfile.ZipFile(zip_path) as zf:
            grouped = defaultdict(dict)
            provenance = {}
            for member in zf.namelist():
                parts = PurePosixPath(member).parts
                sample_part = next((part for part in parts if ID_PATTERN.fullmatch(part)), None)
                if sample_part is None:
                    continue
                grouped.setdefault(sample_part, {})
                basename = PurePosixPath(member).name.lower()
                for layer in LAYERS:
                    if basename.endswith(f"_{layer}.tif") or basename == f"{layer}.tif":
                        grouped[sample_part][layer] = member
                if PurePosixPath(member).name.lower() == "provenance.json":
                    provenance[sample_part] = member
            samples = []
            for sample_id, members in sorted(grouped.items()):
                incident_id, candidate_id = sample_id.rsplit("_", 1)
                profiles, profile_error = {}, None
                for layer, member in members.items():
                    try:
                        profiles[layer] = get_raster_profile(zip_path, member)
                    except Exception as exc:
                        profile_error = f"{layer}: {type(exc).__name__}: {exc}"
                samples.append({
                    "sample_id": sample_id, "incident_id": incident_id,
                    "candidate_id": candidate_id, "members": members,
                    "provenance_member": provenance.get(sample_id),
                    "profiles": profiles, "profile_error": profile_error,
                    "zip_name": archive["path"], "source_folder": archive["source_folder"],
                    **SOURCE_LAYOUT[archive["source_folder"]],
                })
            metadata, metadata_source = member_csv(zf, archive, "metadata")
            status, status_source = member_csv(zf, archive, "status")
            archive["metadata_source"] = metadata_source
            archive["status_source"] = status_source
            archive["samples"] = [s["sample_id"] for s in samples]
            return samples, metadata, status
    finally:
        zip_path.unlink(missing_ok=True)

all_samples, archive_csvs, zip_lookup = [], {}, {}
first_csv_example = None
if INDEX_FILE.exists():
    cached_index = json.loads(INDEX_FILE.read_text())
else:
    cached_index = {}
if cached_index.get("signature") == archive_signature:
    archives = cached_index["archives"]
    all_samples = cached_index["samples"]
    archive_csvs = {
        path: {
            "metadata": pd.DataFrame(payload["metadata"]),
            "status": pd.DataFrame(payload["status"]),
            "metadata_source": payload["metadata_source"],
            "status_source": payload["status_source"],
        }
        for path, payload in cached_index["archive_csvs"].items()
    }
    zip_lookup = {archive["path"]: archive for archive in archives}
    for path, payload in archive_csvs.items():
        if first_csv_example is None and (not payload["metadata"].empty or not payload["status"].empty):
            first_csv_example = (payload["metadata"], payload["status"], path)
    print(f"Loaded cached ZIP index for source revision {SOURCE_REVISION}.")
else:
    for archive in tqdm(archives, desc="Indexing ZIP listings"):
        samples, metadata, status = discover_archive(archive)
        if not samples:
            LOGGER.warning("No sample folders found in %s", archive["path"])
        all_samples.extend(samples)
        zip_lookup[archive["path"]] = archive
        archive_csvs[archive["path"]] = {
            "metadata": metadata, "status": status,
            "metadata_source": archive["metadata_source"],
            "status_source": archive["status_source"],
        }
        if first_csv_example is None and (not metadata.empty or not status.empty):
            first_csv_example = (metadata, status, archive["path"])
    cache_payload = {
        "signature": archive_signature, "archives": archives, "samples": all_samples,
        "archive_csvs": {
            path: {
                "metadata": payload["metadata"].to_dict(orient="records"),
                "status": payload["status"].to_dict(orient="records"),
                "metadata_source": payload["metadata_source"],
                "status_source": payload["status_source"],
            }
            for path, payload in archive_csvs.items()
        },
    }
    temp_index = INDEX_FILE.with_suffix(".tmp")
    temp_index.write_text(json.dumps(cache_payload, default=str))
    os.replace(temp_index, INDEX_FILE)
if not all_samples:
    raise RuntimeError("No sample folders matching incident_candidate were found in the ZIP files.")

print(f"Indexed {len(all_samples):,} sample folders before deduplication.")
index_df = pd.DataFrame([{
    "sample_id": row["sample_id"], "incident_id": row["incident_id"],
    "candidate_id": row["candidate_id"], "label": row["label"],
    "source_id": row["source_id"], "source_folder": row["source_folder"],
    "zip_name": row["zip_name"],
} for row in all_samples])
display(index_df.head(20))
if first_csv_example:
    metadata, status, source_zip = first_csv_example
    print(f"\nCSV inspection from {source_zip}:")
    for title, frame in (("metadata.csv", metadata), ("status.csv", status)):
        print(f"\n{title} columns: {frame.columns.tolist()}")
        print(frame.head().to_string(index=False))
        for column in frame.columns:
            print(f"{title} value counts — {column}:")
            print(frame[column].astype(str).value_counts(dropna=False).head(10).to_string())
else:
    print("No metadata.csv/status.csv was found in the archive or beside it.")


In [ ]:
# Inspect one example sample and summarize all available footprints and native resolutions
example = next((s for s in all_samples if s["source_folder"] == "landslide_early"), all_samples[0])
print(f"Example sample: {example['sample_id']} ({example['zip_name']})")
for layer in LAYERS:
    profile = example["profiles"].get(layer)
    if profile is None:
        print(f"  {layer}: MISSING")
        continue
    bounds = profile["bounds"]
    print(
        f"  {layer}: bands={profile['count']}, shape=({profile['height']}, {profile['width']}), "
        f"dtype={profile['dtypes']}, CRS={profile['crs']}, pixel_size={profile['res']}, "
        f"nodata={profile['nodata']}, bounds={bounds}"
    )
if example.get("provenance_member"):
    example_zip = fetch_zip(example["zip_name"])
    try:
        with zipfile.ZipFile(example_zip) as zf:
            print("\nExample provenance.json:")
            print(zf.read(example["provenance_member"]).decode("utf-8")[:5000])
    finally:
        example_zip.unlink(missing_ok=True)
else:
    print("Example sample has no provenance.json.")

def footprint_metrics(profile):
    if not profile.get("crs"):
        raise ValueError("Missing CRS.")
    left, bottom, right, top = profile["bounds"]
    transformed = transform_bounds(profile["crs"], "EPSG:6933", left, bottom, right, top)
    width_m, height_m = transformed[2] - transformed[0], transformed[3] - transformed[1]
    if width_m <= 0 or height_m <= 0:
        raise ValueError("Invalid footprint.")
    return width_m, height_m, width_m / profile["width"], height_m / profile["height"]

valid_profiles = []
profile_skips = []
for sample in all_samples:
    try:
        if sample["profile_error"]:
            raise ValueError(sample["profile_error"])
        if any(layer not in sample["profiles"] for layer in LAYERS):
            raise ValueError("Missing one or more required TIFFs.")
        values = {layer: footprint_metrics(sample["profiles"][layer]) for layer in LAYERS}
        sample["footprints"] = values
        valid_profiles.append(sample)
    except Exception as exc:
        sample["preflight_error"] = f"{type(exc).__name__}: {exc}"
        profile_skips.append({
            "sample_id": sample["sample_id"], "reason": sample["preflight_error"],
            "zip_name": sample["zip_name"],
        })

if not valid_profiles:
    pd.DataFrame(profile_skips, columns=["sample_id", "reason"]).to_csv(
        WORK / "skipped.csv", index=False
    )
    raise RuntimeError("No sample has all required readable TIFF profiles.")

layer_groups = {
    "planet": ("planet_before", "planet_after"),
    "s1": ("s1_before", "s1_after"),
    "terrain": ("slope", "aspect"),
}
EXPECTED_LAYER_COUNTS = {
    layer: Counter(sample["profiles"][layer]["count"] for sample in valid_profiles).most_common(1)[0][0]
    for layer in LAYERS
}
for sample in valid_profiles:
    mismatches = [
        f"{layer} has {sample['profiles'][layer]['count']} bands; expected {EXPECTED_LAYER_COUNTS[layer]}"
        for layer in LAYERS
        if sample["profiles"][layer]["count"] != EXPECTED_LAYER_COUNTS[layer]
    ]
    if mismatches:
        sample["preflight_error"] = "; ".join(mismatches)
        profile_skips.append({
            "sample_id": sample["sample_id"], "reason": sample["preflight_error"],
            "zip_name": sample["zip_name"],
        })
valid_profiles = [s for s in valid_profiles if not s.get("preflight_error")]
if not valid_profiles:
    pd.DataFrame(profile_skips, columns=["sample_id", "reason"]).to_csv(
        WORK / "skipped.csv", index=False
    )
    raise RuntimeError("No samples have consistent TIFF band counts.")
if EXPECTED_LAYER_COUNTS["s1_before"] != 2 or EXPECTED_LAYER_COUNTS["s1_after"] != 2:
    bad = [s for s in valid_profiles if
           s["profiles"]["s1_before"]["count"] != 2 or s["profiles"]["s1_after"]["count"] != 2]
    profile_skips.extend({
        "sample_id": s["sample_id"],
        "reason": "Sentinel-1 TIFFs must each contain VV and VH (two bands).",
        "zip_name": s["zip_name"],
    } for s in bad)
    pd.DataFrame(profile_skips).to_csv(WORK / "skipped.csv", index=False)
    raise ValueError("Sentinel-1 before/after rasters must each contain VV and VH (two bands).")
if EXPECTED_LAYER_COUNTS["slope"] != 1 or EXPECTED_LAYER_COUNTS["aspect"] != 1:
    bad = [s for s in valid_profiles if
           s["profiles"]["slope"]["count"] != 1 or s["profiles"]["aspect"]["count"] != 1]
    profile_skips.extend({
        "sample_id": s["sample_id"],
        "reason": "Slope and aspect TIFFs must each contain one band.",
        "zip_name": s["zip_name"],
    } for s in bad)
    pd.DataFrame(profile_skips).to_csv(WORK / "skipped.csv", index=False)
    raise ValueError("Slope and aspect TIFFs must each contain exactly one band.")

def common_intersection_footprint(sample):
    bounds = []
    for profile in sample["profiles"].values():
        bounds.append(transform_bounds(profile["crs"], "EPSG:6933",
                                       *profile["bounds"], densify_pts=21))
    left = max(item[0] for item in bounds)
    bottom = max(item[1] for item in bounds)
    right = min(item[2] for item in bounds)
    top = min(item[3] for item in bounds)
    if left >= right or bottom >= top:
        raise ValueError(f"{sample['sample_id']} has no common TIFF footprint.")
    return right - left, top - bottom

for sample in valid_profiles:
    try:
        sample["common_footprint"] = common_intersection_footprint(sample)
    except Exception as exc:
        sample["preflight_error"] = f"{type(exc).__name__}: {exc}"
        profile_skips.append({
            "sample_id": sample["sample_id"], "reason": sample["preflight_error"],
            "zip_name": sample["zip_name"],
        })
valid_profiles = [s for s in valid_profiles if not s.get("preflight_error")]
if not valid_profiles:
    pd.DataFrame(profile_skips, columns=["sample_id", "reason"]).to_csv(
        WORK / "skipped.csv", index=False
    )
    raise RuntimeError("No samples have intersecting modality footprints.")
all_widths = [s["common_footprint"][0] for s in valid_profiles]
all_heights = [s["common_footprint"][1] for s in valid_profiles]
TARGET_FOOTPRINT_M = (float(np.median(all_widths)), float(np.median(all_heights)))
MODALITY_RESOLUTION_M = {}
for modality, layers in layer_groups.items():
    MODALITY_RESOLUTION_M[modality] = (
        float(np.median([s["footprints"][l][2] for s in valid_profiles for l in layers])),
        float(np.median([s["footprints"][l][3] for s in valid_profiles for l in layers])),
    )
MODALITY_SHAPES = {
    modality: (
        max(1, int(round(TARGET_FOOTPRINT_M[1] / resolution[1]))),
        max(1, int(round(TARGET_FOOTPRINT_M[0] / resolution[0]))),
    )
    for modality, resolution in MODALITY_RESOLUTION_M.items()
}
CHANNEL_COUNTS = {}
for modality, layers in layer_groups.items():
    CHANNEL_COUNTS[modality] = sum(EXPECTED_LAYER_COUNTS[layer] for layer in layers)

print(f"\nMedian common footprint (m): {TARGET_FOOTPRINT_M}")
print(f"Chosen fixed shapes (C,H,W): {MODALITY_SHAPES}")
print(f"Median pixel sizes (m): {MODALITY_RESOLUTION_M}")
print(f"Preflight skipped: {len(profile_skips)}")


## 2. Deduplicate and split before pixel processing

The deterministic split keeps every incident in one split, including incidents represented
in multiple source folders. Stratified group folds are used when the data supports them; a
seeded group-aware fallback handles small smoke-test subsets.


In [ ]:
def deduplicate_samples(samples):
    by_id = defaultdict(list)
    for row in samples:
        by_id[row["sample_id"]].append(row)
    kept, log = [], []
    for sample_id, rows in sorted(by_id.items()):
        folders = {r["source_folder"] for r in rows}
        positives = folders & {"landslide_early", "landslide_latest"}
        if positives and "non_landslide" in folders:
            log.append({"sample_id": sample_id, "reason": "id occurs in landslide and non_landslide; dropped from both"})
            continue
        positive_rows = [r for r in rows if r["source_folder"] in {"landslide_early", "landslide_latest"}]
        if len(positive_rows) > 1:
            chosen = sorted(
                positive_rows,
                key=lambda r: (bool(r.get("preflight_error")),
                               r["source_folder"] != "landslide_early", r["zip_name"]),
            )[0]
            log.append({"sample_id": sample_id, "reason": f"duplicate positive source; kept {chosen['source_folder']} from {chosen['zip_name']}"})
            kept.append(chosen)
            continue
        chosen = sorted(
            rows, key=lambda r: (bool(r.get("preflight_error")), r["source_folder"], r["zip_name"])
        )[0]
        if len(rows) > 1:
            log.append({"sample_id": sample_id, "reason": f"duplicate ZIP entry; kept {chosen['zip_name']}"})
        kept.append(chosen)
    return kept, log

samples, dedup_log = deduplicate_samples(all_samples)
print(f"Deduplication decisions logged: {len(dedup_log)}")
samples = [s for s in samples if not s.get("preflight_error")]
if not samples:
    skipped_rows = [
        {"sample_id": item["sample_id"], "reason": item["reason"]}
        for item in dedup_log + profile_skips
    ]
    pd.DataFrame(skipped_rows, columns=["sample_id", "reason"]).to_csv(
        WORK / "skipped.csv", index=False
    )
    raise RuntimeError("No valid samples remain after deduplication and preflight; see skipped.csv.")
if DEBUG_MAX_SAMPLES is not None:
    if DEBUG_MAX_SAMPLES < 6:
        raise ValueError("For a meaningful three-source smoke test, DEBUG_MAX_SAMPLES must be at least 6.")
    rng = np.random.default_rng(SEED)
    groups = defaultdict(list)
    for row in samples:
        groups[row["incident_id"]].append(row)
    source_targets = {
        source: DEBUG_MAX_SAMPLES * sum(r["source_folder"] == source for r in samples) / max(len(samples), 1)
        for source in SOURCE_LAYOUT
    }
    shuffled_groups = list(groups.items())
    rng.shuffle(shuffled_groups)
    chosen_groups, chosen_count = [], Counter()
    while shuffled_groups:
        remaining = []
        progressed = False
        for incident_id, rows in shuffled_groups:
            if len(chosen_groups) + len(rows) > DEBUG_MAX_SAMPLES:
                remaining.append((incident_id, rows))
                continue
            contribution = Counter(r["source_folder"] for r in rows)
            deficit = sum(max(source_targets[s] - chosen_count[s], 0) * contribution[s] for s in SOURCE_LAYOUT)
            if len(chosen_groups) < DEBUG_MAX_SAMPLES and (deficit > 0 or not chosen_groups):
                chosen_groups.extend(rows)
                chosen_count.update(contribution)
                progressed = True
            else:
                remaining.append((incident_id, rows))
        shuffled_groups = remaining
        if not progressed:
            break
    if len(chosen_groups) < 3 or set(r["source_folder"] for r in chosen_groups) != set(SOURCE_LAYOUT):
        raise ValueError("DEBUG_MAX_SAMPLES must include all three source folders and at least three groups; increase it.")
    samples = chosen_groups[:DEBUG_MAX_SAMPLES]
    print(f"DEBUG_MAX_SAMPLES selected {len(samples)} samples: {Counter(r['source_folder'] for r in samples)}")

def group_fallback(rows, seed):
    """Greedily minimize per-source and total-count deviations without splitting groups."""
    rng = np.random.default_rng(seed)
    by_group = defaultdict(list)
    for row in rows:
        by_group[row["incident_id"]].append(row)
    group_items = list(by_group.items())
    rng.shuffle(group_items)
    group_items.sort(key=lambda item: (-len(item[1]), item[0]))
    names = ["train", "test", "val"]
    fractions = np.array([SPLIT[n] for n in names], dtype=float)
    if len(group_items) < len(names):
        raise ValueError("At least three distinct incidents are required for train/test/val.")
    totals = Counter(r["source_folder"] for r in rows)
    assigned = {n: Counter() for n in names}
    group_assignment = {}
    for target, (incident_id, group_rows) in zip(names, group_items[:len(names)]):
        assigned[target].update(Counter(r["source_folder"] for r in group_rows))
        group_assignment[incident_id] = target
    for incident_id, group_rows in group_items[len(names):]:
        contribution = Counter(r["source_folder"] for r in group_rows)
        best = None
        for target in names:
            trial = {n: assigned[n].copy() for n in names}
            trial[target].update(contribution)
            loss = 0.0
            for folder, total in totals.items():
                for n, frac in zip(names, fractions):
                    desired = total * frac
                    loss += ((trial[n][folder] - desired) / max(desired, 1.0)) ** 2
            for n, frac in zip(names, fractions):
                desired = len(rows) * frac
                actual = sum(trial[n].values())
                loss += ((actual - desired) / max(desired, 1.0)) ** 2
            candidate = (loss, names.index(target), target)
            if best is None or candidate < best:
                best = candidate
        target = best[2]
        assigned[target].update(contribution)
        group_assignment[incident_id] = target
    return group_assignment

def stratified_group_assignment(rows):
    """Assign 70/20/10 groups using nested stratified grouped folds when feasible."""
    y = np.array([r["source_id"] for r in rows])
    groups = np.array([r["incident_id"] for r in rows])
    names = np.array(["train", "test", "val"])
    unique_by_source = {
        source_id: len({r["incident_id"] for r in rows if r["source_id"] == source_id})
        for source_id in range(3)
    }
    if min(unique_by_source.values()) >= 5:
        splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
        candidates = list(splitter.split(np.zeros(len(rows)), y, groups))
        test_idx, rest_idx = min(
            candidates,
            key=lambda pair: sum(abs(np.mean(y[pair[1]] == sid) - np.mean(y == sid)) for sid in range(3))
                            + abs(len(pair[1]) / len(rows) - 0.2),
        )
        test_idx, rest_idx = rest_idx, test_idx
        rest_rows = [rows[i] for i in rest_idx]
        rest_y = np.array([r["source_id"] for r in rest_rows])
        rest_groups = np.array([r["incident_id"] for r in rest_rows])
        remaining_groups = min(len({r["incident_id"] for r in rest_rows if r["source_id"] == sid}) for sid in range(3))
        if remaining_groups >= 8:
            val_splitter = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=SEED + 1)
            val_candidates = list(val_splitter.split(np.zeros(len(rest_rows)), rest_y, rest_groups))
            train_local, val_local = min(
                val_candidates,
                key=lambda pair: sum(abs(np.mean(rest_y[pair[0]] == sid) - np.mean(rest_y == sid)) for sid in range(3))
                                + abs(len(pair[0]) / len(rest_rows) - 0.125),
            )
            mapping = {rows[i]["incident_id"]: "test" for i in test_idx}
            mapping.update({rest_rows[i]["incident_id"]: "val" for i in val_local})
            mapping.update({rest_rows[i]["incident_id"]: "train" for i in train_local})
            return mapping
    return group_fallback(rows, SEED)

group_split = stratified_group_assignment(samples)
for row in samples:
    row["split"] = group_split[row["incident_id"]]

incident_sources = defaultdict(set)
for row in samples:
    incident_sources[row["incident_id"]].add(row["source_folder"])
cross_folder_incidents = sorted(
    incident_id for incident_id, folders in incident_sources.items() if len(folders) > 1
)
for incident_id in cross_folder_incidents:
    LOGGER.info(
        "Incident %s occurs in source folders %s; all its samples stay in %s.",
        incident_id, sorted(incident_sources[incident_id]), group_split[incident_id],
    )

split_counts = pd.crosstab(
    pd.Series([r["split"] for r in samples], name="split"),
    pd.Series([r["source_folder"] for r in samples], name="source_folder"),
).reindex(index=["train", "test", "val"], columns=list(SOURCE_LAYOUT), fill_value=0)
print("Counts per split x source folder:")
display(split_counts)
print("Counts per split x label:")
display(pd.crosstab(
    pd.Series([r["split"] for r in samples], name="split"),
    pd.Series([r["label"] for r in samples], name="label"),
).reindex(index=["train", "test", "val"], fill_value=0))
incident_sets = {n: {r["incident_id"] for r in samples if r["split"] == n} for n in SPLIT}
for a, b in (("train", "test"), ("train", "val"), ("test", "val")):
    overlap = incident_sets[a] & incident_sets[b]
    assert not overlap, f"Incident leakage between {a} and {b}: {sorted(overlap)[:10]}"
for source in SOURCE_LAYOUT:
    test_count = int(split_counts.loc["test", source])
    if test_count < 30:
        print(f"WARNING: {source} has only {test_count} test samples; metrics may be unreliable.")
for split_name in SPLIT:
    missing_sources = [source for source in SOURCE_LAYOUT if split_counts.loc[split_name, source] == 0]
    if missing_sources:
        print(f"WARNING: {split_name} has no samples from {missing_sources}; increase dataset/debug size.")


## 3. Calibrate Sentinel-1 and finalize shapes

Positive-only Sentinel-1 values are ambiguous (linear power versus amplitude); `auto` assumes
linear power and uses `10*log10`. Override `S1_INPUT_SCALE` when provenance or producer
documentation identifies the encoding. Per-channel clip bounds are estimated once from
deterministically selected samples and applied unchanged to every split.


In [ ]:
def extract_sample_files(zf, sample, temp_dir):
    """Extract only the selected sample's TIFFs and provenance into a temporary directory."""
    out = {}
    wanted = dict(sample["members"])
    if sample.get("provenance_member"):
        wanted["provenance"] = sample["provenance_member"]
    for key, member in wanted.items():
        target = Path(temp_dir) / f"{sample['sample_id']}_{key}_{PurePosixPath(member).name}"
        with zf.open(member) as source, target.open("wb") as destination:
            shutil.copyfileobj(source, destination)
        out[key] = target
    return out

def read_crop(path, expected_profile, shape_hw, center_6933):
    """Read, crop/pad around the shared map center, mask nodata, and preserve native pixels."""
    out_h, out_w = shape_hw
    with rasterio.open(path) as ds:
        if ds.crs is None or expected_profile["crs"] != ds.crs.to_string():
            raise ValueError(f"CRS changed since inspection for {path.name}.")
        cx, cy = transform("EPSG:6933", ds.crs, [center_6933[0]], [center_6933[1]])
        row_center, col_center = ds.index(cx[0], cy[0])
        row0, col0 = row_center - out_h // 2, col_center - out_w // 2
        row1, col1 = row0 + out_h, col0 + out_w
        src_r0, src_c0 = max(row0, 0), max(col0, 0)
        src_r1, src_c1 = min(row1, ds.height), min(col1, ds.width)
        data = np.zeros((ds.count, out_h, out_w), dtype=np.float32)
        valid = np.zeros((ds.count, out_h, out_w), dtype=bool)
        if src_r1 > src_r0 and src_c1 > src_c0:
            block = ds.read(window=((src_r0, src_r1), (src_c0, src_c1))).astype(np.float32, copy=False)
            mask = np.isfinite(block)
            if ds.nodata is not None:
                mask &= block != ds.nodata
            dst_r0, dst_c0 = src_r0 - row0, src_c0 - col0
            dst_r1, dst_c1 = dst_r0 + block.shape[1], dst_c0 + block.shape[2]
            data[:, dst_r0:dst_r1, dst_c0:dst_c1] = np.where(mask, block, 0)
            valid[:, dst_r0:dst_r1, dst_c0:dst_c1] = mask
        return data, valid

def sample_center_and_validate(sample):
    """Return the common intersection center and reject misaligned/native-scale outliers."""
    transformed_bounds = []
    centers = []
    for layer in LAYERS:
        profile = sample["profiles"][layer]
        left, bottom, right, top = transform_bounds(
            profile["crs"], "EPSG:6933", *profile["bounds"], densify_pts=21
        )
        transformed_bounds.append((left, bottom, right, top))
    intersection = (
        max(b[0] for b in transformed_bounds), max(b[1] for b in transformed_bounds),
        min(b[2] for b in transformed_bounds), min(b[3] for b in transformed_bounds),
    )
    if intersection[0] >= intersection[2] or intersection[1] >= intersection[3]:
        raise ValueError("Required modalities do not have a common ground footprint.")
    center = ((intersection[0] + intersection[2]) / 2, (intersection[1] + intersection[3]) / 2)
    for layer in LAYERS:
        width_m, height_m, pixel_w, pixel_h = sample["footprints"][layer]
        modality = "planet" if layer.startswith("planet") else "s1" if layer.startswith("s1") else "terrain"
        if abs(pixel_w / MODALITY_RESOLUTION_M[modality][0] - 1) > SHAPE_RESOLUTION_TOLERANCE:
            raise ValueError(f"{layer} pixel width differs from median by over {SHAPE_RESOLUTION_TOLERANCE:.0%}.")
        if abs(pixel_h / MODALITY_RESOLUTION_M[modality][1] - 1) > SHAPE_RESOLUTION_TOLERANCE:
            raise ValueError(f"{layer} pixel height differs from median by over {SHAPE_RESOLUTION_TOLERANCE:.0%}.")
        crop_width_m = MODALITY_SHAPES[modality][1] * pixel_w
        crop_height_m = MODALITY_SHAPES[modality][0] * pixel_h
        if (abs(crop_width_m / TARGET_FOOTPRINT_M[0] - 1) > SHAPE_RESOLUTION_TOLERANCE
                or abs(crop_height_m / TARGET_FOOTPRINT_M[1] - 1) > SHAPE_RESOLUTION_TOLERANCE):
            raise ValueError(f"{layer} crop footprint differs from the target by over "
                             f"{SHAPE_RESOLUTION_TOLERANCE:.0%}.")
    return center

def select_provenance_fields(obj, prefix="provenance"):
    """Flatten a small set of date, coordinate, sensor, and resolution provenance fields."""
    selected, stack = {}, [(prefix, obj)]
    tokens = ("date", "time", "lat", "lon", "coord", "sensor", "platform",
              "orbit", "resolution", "pixel", "source")
    while stack:
        path, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f"{path}_{key}", child))
        elif isinstance(value, list):
            if len(value) <= 10 and any(token in path.lower() for token in tokens):
                selected[path] = value
        elif any(token in path.lower() for token in tokens):
            selected[path] = value
    return selected

def match_csv_row(frame, sample):
    """Select a metadata/status row by sample ID or incident/candidate pair."""
    if frame.empty:
        return {}
    lower = {str(c).strip().lower(): c for c in frame.columns}
    incident_col = next((lower[k] for k in ("incident_id", "incident", "incidentid") if k in lower), None)
    candidate_col = next((lower[k] for k in ("candidate_id", "candidate", "candidateid") if k in lower), None)
    sample_col = next((lower[k] for k in (
        "sample_id", "sample", "folder", "tile_id", "candidate_folder", "folder_name", "name"
    ) if k in lower), None)
    matched = pd.Series(False, index=frame.index)
    if incident_col and candidate_col:
        matched = (
            frame[incident_col].astype(str).str.replace(r"\.0$", "", regex=True).eq(sample["incident_id"])
            & frame[candidate_col].astype(str).str.replace(r"\.0$", "", regex=True).eq(sample["candidate_id"])
        )
    elif sample_col:
        matched = frame[sample_col].astype(str).eq(sample["sample_id"])
    elif candidate_col and frame[candidate_col].astype(str).nunique() == len(frame):
        matched = frame[candidate_col].astype(str).str.replace(r"\.0$", "", regex=True).eq(sample["candidate_id"])
    if not matched.any():
        for column in frame.columns:
            values = frame[column].astype(str).str.strip()
            candidate_match = (
                values.eq(sample["sample_id"])
                | values.str.endswith("/" + sample["sample_id"])
                | values.str.endswith("\\" + sample["sample_id"])
            )
            if candidate_match.any():
                matched |= candidate_match
                break
    rows = frame.loc[matched]
    return rows.iloc[0].to_dict() if len(rows) else {}

def read_archive_csv(zf, archive, kind):
    """Read per-archive metadata/status, with the inspected external CSV as a fallback."""
    member = next((m for m in zf.namelist() if PurePosixPath(m).name.lower() == f"{kind}.csv"), None)
    if member:
        with zf.open(member) as stream:
            return pd.read_csv(stream)
    cached = archive_csvs[archive["path"]][kind]
    if not cached.empty:
        return cached
    return pd.DataFrame()

calibration_rows = sorted(samples, key=lambda r: hashlib.sha256(
    f"{SEED}:{r['sample_id']}".encode()
).hexdigest())[:S1_CALIBRATION_SAMPLES]
calibration_values = defaultdict(list)
positive_s1_values = []
for sample in tqdm(calibration_rows, desc="Calibrating Sentinel-1"):
    zip_path = fetch_zip(sample["zip_name"])
    try:
        with tempfile.TemporaryDirectory(dir=WORK) as temp_dir, zipfile.ZipFile(zip_path) as zf:
            paths = extract_sample_files(zf, sample, temp_dir)
            for layer in ("s1_before", "s1_after"):
                with rasterio.open(paths[layer]) as ds:
                    raw = ds.read().astype(np.float32, copy=False)
                    valid = np.isfinite(raw)
                    if ds.nodata is not None:
                        valid &= raw != ds.nodata
                    vals = raw[valid]
                    if vals.size:
                        positive_s1_values.append(vals[::max(1, vals.size // 2048)])
                        for band in range(raw.shape[0]):
                            band_vals = raw[band][valid[band]]
                            if band_vals.size:
                                calibration_values[f"{layer}_{band}"].append(
                                    band_vals[::max(1, band_vals.size // 4096)]
                                )
    finally:
        zip_path.unlink(missing_ok=True)

all_s1 = np.concatenate(positive_s1_values) if positive_s1_values else np.array([])
if S1_INPUT_SCALE == "auto":
    S1_SCALE_DECISION = "db" if all_s1.size and np.nanmin(all_s1) < 0 else "linear_power"
else:
    S1_SCALE_DECISION = S1_INPUT_SCALE
print(f"Sentinel-1 scale decision: {S1_SCALE_DECISION} (configured={S1_INPUT_SCALE!r}).")
if S1_INPUT_SCALE == "auto" and S1_SCALE_DECISION == "linear_power":
    print("Auto-detection cannot distinguish power from amplitude; verify the producer's encoding.")

def convert_s1(values):
    if S1_SCALE_DECISION == "db":
        return values
    if S1_SCALE_DECISION == "linear_power":
        return 10.0 * np.log10(np.maximum(values, np.finfo(np.float32).tiny))
    return 20.0 * np.log10(np.maximum(values, np.finfo(np.float32).tiny))

S1_CLIP_BOUNDS = {}
for layer in ("s1_before", "s1_after"):
    band_count = CHANNEL_COUNTS["s1"] // 2
    for band in range(band_count):
        values = calibration_values.get(f"{layer}_{band}", [])
        if not values:
            raise RuntimeError(f"No valid calibration values for {layer} band {band}.")
        raw_values = np.concatenate(values)
        if S1_SCALE_DECISION != "db":
            raw_values = raw_values[raw_values > 0]
        if not raw_values.size:
            raise RuntimeError(f"No positive linear calibration values for {layer} band {band}.")
        converted = convert_s1(raw_values)
        lo, hi = np.percentile(converted, S1_CLIP_PERCENTILES)
        S1_CLIP_BOUNDS[f"{layer}_{band}"] = [float(lo), float(hi)]
print("Sentinel-1 global per-channel clip bounds (dB):")
print(json.dumps(S1_CLIP_BOUNDS, indent=2))

preprocessing_decisions = {
    "s1_input_scale": S1_SCALE_DECISION,
    "s1_auto_detection_note": "Positive-only values default to linear power; confirm producer encoding.",
    "s1_clip_percentiles": list(S1_CLIP_PERCENTILES),
    "s1_clip_bounds_db": S1_CLIP_BOUNDS,
    "target_footprint_m": list(TARGET_FOOTPRINT_M),
    "modality_shapes": {k: list(v) for k, v in MODALITY_SHAPES.items()},
    "median_pixel_size_m": {k: list(v) for k, v in MODALITY_RESOLUTION_M.items()},
    "shape_resolution_tolerance": SHAPE_RESOLUTION_TOLERANCE,
    "nodata": "masked from TIFF nodata/non-finite values, then zero-filled; padding is zero",
    "resampling": "none; crop/pad around shared EPSG:6933 intersection center",
    "aspect": "degrees converted to sin/cos; nodata or slope<=0.01 has sin=cos=0 and mask=1",
}
print("Preprocessing choices:")
print(json.dumps(preprocessing_decisions, indent=2))


## 4. Stream samples into resumable staging memmaps

One archive is downloaded at a time. Only one sample's six rasters are extracted at a time;
after it is written, its temporary files are removed. A failure is recorded in `skipped.csv`
and processing continues. `progress.json` is atomically updated after each successful sample.


In [ ]:
def make_structured_dtype():
    float_dtype = np.dtype(OUTPUT_DTYPE)
    return np.dtype([
        ("planet", float_dtype, (CHANNEL_COUNTS["planet"], *MODALITY_SHAPES["planet"])),
        ("s1", float_dtype, (CHANNEL_COUNTS["s1"], *MODALITY_SHAPES["s1"])),
        ("terrain", float_dtype, (4, *MODALITY_SHAPES["terrain"])),
        ("label", np.int8), ("source_id", np.int8), ("sample_idx", np.int32),
    ])

ARRAY_DTYPE = make_structured_dtype()
split_candidates = {
    name: sorted([s for s in samples if s["split"] == name],
                 key=lambda r: hashlib.sha256(f"{SEED}:{name}:{r['sample_id']}".encode()).hexdigest())
    for name in SPLIT
}
if any(not split_candidates[name] for name in SPLIT):
    raise ValueError("At least one split is empty; increase the dataset or debug sample count.")
candidate_locations = {
    row["sample_id"]: (split, index)
    for split, rows in split_candidates.items()
    for index, row in enumerate(rows)
}

estimated_record_bytes = ARRAY_DTYPE.itemsize
estimated_staging_bytes = sum(len(rows) for rows in split_candidates.values()) * estimated_record_bytes
estimated_output_bytes = {
    f"{split}.npy": len(rows) * estimated_record_bytes
    for split, rows in split_candidates.items()
}
for source_folder in SOURCE_LAYOUT:
    count = sum(row["source_folder"] == source_folder for row in split_candidates["test"])
    estimated_output_bytes[f"test_{source_folder}.npy"] = count * estimated_record_bytes
estimated_test_duplicate_bytes = sum(
    size for name, size in estimated_output_bytes.items() if name.startswith("test_")
)
estimated_peak_bytes = estimated_staging_bytes + estimated_staging_bytes + estimated_test_duplicate_bytes
free_bytes = shutil.disk_usage(WORK).free
print("Estimated final .npy sizes:")
for filename, size in estimated_output_bytes.items():
    print(f"  {filename}: {size / 1e9:.3f} GB")
print(f"Estimated peak including compaction and per-source test copies: {estimated_peak_bytes / 1e9:.2f} GB")
print(f"Free disk: {free_bytes / 1e9:.2f} GB")
if estimated_peak_bytes > free_bytes:
    print("WARNING: estimated peak exceeds current free disk; reduce DEBUG_MAX_SAMPLES or use float16.")

signature_payload = {
    "pipeline_version": 2,
    "repo": SOURCE_REPO, "source_revision": SOURCE_REVISION,
    "archive_signature": archive_signature,
    "samples": [(s["sample_id"], s["zip_name"], s["source_folder"], s["split"]) for s in samples],
    "debug": DEBUG_MAX_SAMPLES, "dtype": OUTPUT_DTYPE, "shapes": MODALITY_SHAPES,
    "scale": S1_SCALE_DECISION, "bounds": S1_CLIP_BOUNDS,
    "clip_percentiles": S1_CLIP_PERCENTILES, "split_fractions": SPLIT,
    "shape_tolerance": SHAPE_RESOLUTION_TOLERANCE, "seed": SEED,
}
RUN_SIGNATURE = hashlib.sha256(json.dumps(signature_payload, sort_keys=True).encode()).hexdigest()
PROGRESS_PATH = WORK / "progress.json"
SKIPPED_PATH = WORK / "skipped.csv"
if PROGRESS_PATH.exists():
    progress = json.loads(PROGRESS_PATH.read_text())
    if progress.get("signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "WORK_DIR contains a different run signature. Use a fresh WORK_DIR for changed "
            "data/configuration; do not delete results you need."
        )
else:
    progress = {
        "signature": RUN_SIGNATURE, "done_sample_ids": [], "skipped": [],
        "sample_metadata": {}, "stats": {}, "completed_zips": [],
    }
    progress["skipped"] = []
if progress.get("finalized"):
    raise RuntimeError(
        "This WORK_DIR already has finalized arrays. Start execution at the verification "
        "cell, or at the upload cell if verification already passed."
    )

def save_progress():
    temp = PROGRESS_PATH.with_suffix(".tmp")
    temp.write_text(json.dumps(progress, default=str))
    os.replace(temp, PROGRESS_PATH)

done_ids = set(progress["done_sample_ids"])
skipped_by_id = {row["sample_id"]: row for row in progress.get("skipped", [])}
stage_maps = {}
for split, rows in split_candidates.items():
    stage_path = WORK / f".stage_{split}.npy"
    expected_shape = (len(rows),)
    if stage_path.exists():
        mm = np.lib.format.open_memmap(stage_path, mode="r+")
        if mm.shape != expected_shape or mm.dtype != ARRAY_DTYPE:
            raise RuntimeError(f"Staging file mismatch: {stage_path}")
    else:
        mm = np.lib.format.open_memmap(stage_path, mode="w+", dtype=ARRAY_DTYPE, shape=expected_shape)
    stage_maps[split] = mm

def safe_value(value):
    if value is None:
        return None
    if isinstance(value, (dict, list, tuple, np.ndarray)):
        return json.dumps(value, default=str)
    if pd.api.types.is_scalar(value) and pd.isna(value):
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)

def preprocess_sample(sample, files):
    """Create three fixed-shape modalities and nodata fractions from one sample's TIFFs."""
    center = sample_center_and_validate(sample)
    layer_data, layer_valid = {}, {}
    for layer in LAYERS:
        modality = "planet" if layer.startswith("planet") else "s1" if layer.startswith("s1") else "terrain"
        data, valid = read_crop(files[layer], sample["profiles"][layer], MODALITY_SHAPES[modality], center)
        layer_data[layer], layer_valid[layer] = data, valid
    planet = np.concatenate([layer_data["planet_before"], layer_data["planet_after"]], axis=0)
    planet_valid = np.concatenate([layer_valid["planet_before"], layer_valid["planet_after"]], axis=0)
    s1_chunks, s1_valid_chunks = [], []
    band_count = CHANNEL_COUNTS["s1"] // 2
    for layer in ("s1_before", "s1_after"):
        values = layer_data[layer].copy()
        valid = layer_valid[layer] & np.isfinite(values)
        if S1_SCALE_DECISION != "db":
            valid &= values > 0
            values = convert_s1(values)
        for band in range(values.shape[0]):
            lo, hi = S1_CLIP_BOUNDS[f"{layer}_{band}"]
            values[band] = np.clip(values[band], lo, hi)
            values[band][~valid[band]] = 0
        s1_chunks.append(values)
        s1_valid_chunks.append(valid)
    s1 = np.concatenate(s1_chunks, axis=0)
    s1_valid = np.concatenate(s1_valid_chunks, axis=0)

    slope = layer_data["slope"][0]
    slope_valid = layer_valid["slope"][0]
    aspect = layer_data["aspect"][0]
    aspect_valid = layer_valid["aspect"][0]
    flat = (~aspect_valid) | (~slope_valid) | (slope <= 0.01)
    radians = np.deg2rad(np.mod(aspect, 360.0))
    sin_aspect = np.where(flat, 0, np.sin(radians))
    cos_aspect = np.where(flat, 0, np.cos(radians))
    terrain = np.stack([slope, sin_aspect, cos_aspect, flat.astype(np.float32)], axis=0)
    terrain_valid = np.stack([slope_valid, aspect_valid, aspect_valid, np.ones_like(flat)], axis=0)
    nodata = {
        "planet": float(1 - planet_valid.mean()),
        "s1": float(1 - s1_valid.mean()),
        "terrain": float(1 - np.concatenate(
            [layer_valid["slope"], layer_valid["aspect"]], axis=0
        ).mean()),
    }
    return {
        "planet": planet.astype(OUTPUT_DTYPE), "s1": s1.astype(OUTPUT_DTYPE),
        "terrain": terrain.astype(OUTPUT_DTYPE),
        "valid": {"planet": planet_valid, "s1": s1_valid, "terrain": terrain_valid},
        "nodata": nodata,
    }

def init_stat_bucket():
    return {
        key: {"sum": np.zeros(count, dtype=np.float64).tolist(),
              "sum_sq": np.zeros(count, dtype=np.float64).tolist(),
              "count": np.zeros(count, dtype=np.int64).tolist()}
        for key, count in (("planet", CHANNEL_COUNTS["planet"]),
                           ("s1", CHANNEL_COUNTS["s1"]), ("terrain", 4))
    }

if not progress["stats"]:
    progress["stats"] = init_stat_bucket()

def add_train_stats(processed):
    """Accumulate per-channel moments only over valid pixels from training samples."""
    for modality in ("planet", "s1", "terrain"):
        values = processed[modality].astype(np.float64, copy=False)
        valid = processed["valid"][modality]
        bucket = progress["stats"][modality]
        for channel in range(values.shape[0]):
            selected = values[channel][valid[channel]]
            bucket["sum"][channel] += float(selected.sum(dtype=np.float64))
            bucket["sum_sq"][channel] += float(np.square(selected).sum(dtype=np.float64))
            bucket["count"][channel] += int(selected.size)

for archive in tqdm(archives, desc="Processing ZIPs"):
    rows = [s for s in samples if s["zip_name"] == archive["path"]]
    pending = [s for s in rows if s["sample_id"] not in done_ids and s["sample_id"] not in skipped_by_id]
    if not pending:
        continue
    zip_path = fetch_zip(archive["path"])
    try:
        with tempfile.TemporaryDirectory(dir=WORK) as temp_dir, zipfile.ZipFile(zip_path) as zf:
            metadata_frame = read_archive_csv(zf, archive, "metadata")
            status_frame = read_archive_csv(zf, archive, "status")
            for sample in tqdm(pending, desc=Path(archive["path"]).name, leave=False):
                try:
                    if sample.get("preflight_error"):
                        raise ValueError(sample["preflight_error"])
                    with tempfile.TemporaryDirectory(dir=temp_dir) as sample_dir:
                        files = extract_sample_files(zf, sample, sample_dir)
                        processed = preprocess_sample(sample, files)
                        for field in ("planet", "s1", "terrain"):
                            expected_shape = ARRAY_DTYPE.fields[field][0].shape
                            if processed[field].shape != expected_shape:
                                raise ValueError(
                                    f"{field} shape {processed[field].shape} != expected {expected_shape}"
                                )
                        split, row_index = candidate_locations[sample["sample_id"]]
                        target = stage_maps[split][row_index]
                        for field in ("planet", "s1", "terrain"):
                            target[field] = processed[field]
                        target["label"] = sample["label"]
                        target["source_id"] = sample["source_id"]
                        target["sample_idx"] = row_index
                        provenance = {}
                        if "provenance" in files:
                            provenance = select_provenance_fields(json.loads(files["provenance"].read_text()))
                        metadata_row = match_csv_row(metadata_frame, sample)
                        status_row = match_csv_row(status_frame, sample)
                        manifest = {
                            "sample_id": sample["sample_id"], "incident_id": sample["incident_id"],
                            "candidate_id": sample["candidate_id"], "label": sample["label"],
                            "source_id": sample["source_id"], "source_folder": sample["source_folder"],
                            "zip_name": sample["zip_name"], "split": split,
                            **{f"nodata_fraction_{k}": v for k, v in processed["nodata"].items()},
                            **{f"metadata_{k}": safe_value(v) for k, v in metadata_row.items()},
                            **{f"status_{k}": safe_value(v) for k, v in status_row.items()},
                            **{k: safe_value(v) for k, v in provenance.items()},
                        }
                        old_stats = json.loads(json.dumps(progress["stats"]))
                        try:
                            if split == "train":
                                add_train_stats(processed)
                            progress["sample_metadata"][sample["sample_id"]] = manifest
                            stage_maps[split].flush()
                            done_ids.add(sample["sample_id"])
                            progress["done_sample_ids"] = sorted(done_ids)
                            save_progress()
                        except Exception:
                            progress["stats"] = old_stats
                            progress["sample_metadata"].pop(sample["sample_id"], None)
                            done_ids.discard(sample["sample_id"])
                            progress["done_sample_ids"] = sorted(done_ids)
                            raise
                except Exception as exc:
                    LOGGER.exception("Skipping sample %s", sample["sample_id"])
                    reason = f"{type(exc).__name__}: {exc}"
                    skipped_by_id[sample["sample_id"]] = {
                        "sample_id": sample["sample_id"], "reason": reason,
                        "zip_name": sample["zip_name"],
                    }
                    progress["skipped"] = list(skipped_by_id.values())
                    save_progress()
    finally:
        zip_path.unlink(missing_ok=True)

for stage_map in stage_maps.values():
    stage_map.flush()
progress["completed_zips"] = sorted({s["zip_name"] for s in samples
                                    if s["sample_id"] in done_ids or s["sample_id"] in skipped_by_id})
progress["skipped"] = list(skipped_by_id.values())
save_progress()
skip_log_rows = [
    {"sample_id": item["sample_id"], "reason": item["reason"], "zip_name": item.get("zip_name", "")}
    for item in dedup_log + profile_skips
] + list(skipped_by_id.values())
pd.DataFrame(skip_log_rows, columns=["sample_id", "reason", "zip_name"]).to_csv(SKIPPED_PATH, index=False)
print(f"Processed successfully: {len(done_ids):,}; skipped: {len(skipped_by_id):,}")


## 5. Compact arrays, manifests, and train statistics

Rows remain in a seeded order within each split. `sample_idx` is the row index into its
matching manifest. Per-source test arrays have their own matching manifest and index values.


In [ ]:
def open_stage(split):
    return np.lib.format.open_memmap(WORK / f".stage_{split}.npy", mode="r")

def manifest_frame(records):
    """Keep arbitrary source metadata parquet-safe while preserving primary typed fields."""
    frame = pd.DataFrame(records)
    for column in frame.columns:
        if column.startswith(("metadata_", "status_", "provenance_")):
            frame[column] = frame[column].map(
                lambda value: None if value is None or pd.isna(value) else str(value)
            )
    return frame

successful_by_split = {
    name: [row for row in split_candidates[name] if row["sample_id"] in done_ids]
    for name in SPLIT
}
if any(not successful_by_split[name] for name in SPLIT):
    raise RuntimeError("A split has no successful samples; inspect skipped.csv and source TIFFs.")

manifest_frames = {}
for split, rows in successful_by_split.items():
    stage = open_stage(split)
    output_path = WORK / f"{split}.npy"
    output = np.lib.format.open_memmap(
        output_path, mode="w+", dtype=ARRAY_DTYPE, shape=(len(rows),)
    )
    records = []
    for final_index, row in enumerate(tqdm(rows, desc=f"Compacting {split}")):
        _, stage_index = candidate_locations[row["sample_id"]]
        output[final_index] = stage[stage_index]
        output[final_index]["sample_idx"] = final_index
        manifest = dict(progress["sample_metadata"][row["sample_id"]])
        manifest["row_index"] = final_index
        records.append(manifest)
    output.flush()
    del output, stage
    frame = manifest_frame(records)
    frame.to_parquet(WORK / f"manifest_{split}.parquet", index=False)
    manifest_frames[split] = frame

test_full = np.load(WORK / "test.npy", mmap_mode="r")
test_manifest = manifest_frames["test"]
for source_folder in SOURCE_LAYOUT:
    subset_rows = np.flatnonzero(test_manifest["source_folder"].to_numpy() == source_folder)
    source_path = WORK / f"test_{source_folder}.npy"
    subset = np.lib.format.open_memmap(
        source_path, mode="w+", dtype=ARRAY_DTYPE, shape=(len(subset_rows),)
    )
    source_manifest = test_manifest.iloc[subset_rows].copy().reset_index(drop=True)
    for i, test_row in enumerate(subset_rows):
        subset[i] = test_full[test_row]
        subset[i]["sample_idx"] = i
    subset.flush()
    source_manifest["row_index"] = np.arange(len(source_manifest), dtype=np.int32)
    source_manifest.to_parquet(WORK / f"manifest_test_{source_folder}.parquet", index=False)
    del subset
del test_full

def finalize_stats():
    stats = {}
    for modality, bucket in progress["stats"].items():
        count = np.asarray(bucket["count"], dtype=np.float64)
        if np.any(count == 0):
            raise RuntimeError(f"No valid training pixels for one or more {modality} channels.")
        mean = np.asarray(bucket["sum"], dtype=np.float64) / count
        variance = np.maximum(np.asarray(bucket["sum_sq"], dtype=np.float64) / count - mean**2, 0)
        std = np.sqrt(variance)
        std[std == 0] = 1.0
        stats[modality] = {"mean": mean.tolist(), "std": std.tolist(),
                           "valid_pixel_count": count.astype(np.int64).tolist()}
    return stats

train_stats = finalize_stats()
planet_names = [f"planet_before_band_{i + 1}" for i in range(CHANNEL_COUNTS["planet"] // 2)]
planet_names += [f"planet_after_band_{i + 1}" for i in range(CHANNEL_COUNTS["planet"] // 2)]
s1_names = [f"s1_before_band_{i + 1}" for i in range(CHANNEL_COUNTS["s1"] // 2)]
s1_names += [f"s1_after_band_{i + 1}" for i in range(CHANNEL_COUNTS["s1"] // 2)]
def source_band_names(layer, prefix):
    profile = next(s["profiles"][layer] for s in valid_profiles)
    descriptions = profile["descriptions"]
    return [
        f"{prefix}_{description.strip()}" if description else f"{prefix}_band_{index + 1}"
        for index, description in enumerate(descriptions)
    ]

planet_names = (
    source_band_names("planet_before", "planet_before")
    + source_band_names("planet_after", "planet_after")
)
s1_names = (
    source_band_names("s1_before", "s1_before")
    + source_band_names("s1_after", "s1_after")
)
dataset_info = {
    "source_revision": SOURCE_REVISION,
    "label_map": {"1": "landslide", "0": "non_landslide"},
    "source_map": {"0": "landslide_early", "1": "landslide_latest", "2": "non_landslide"},
    "fields": {name: {"dtype": str(ARRAY_DTYPE.fields[name][0]),
                      "shape": list(ARRAY_DTYPE.fields[name][0].shape or [])}
               for name in ARRAY_DTYPE.names},
    "record_dtype": ARRAY_DTYPE.descr,
    "channel_names": {
        "planet": planet_names, "s1": s1_names,
        "terrain": ["slope", "sin_aspect", "cos_aspect", "flat_or_undefined_aspect"],
    },
    "pixel_size_m": {k: list(v) for k, v in MODALITY_RESOLUTION_M.items()},
    "split_sizes": {k: len(v) for k, v in successful_by_split.items()},
    "cross_folder_incidents_kept_in_one_split": cross_folder_incidents,
    "counts_per_split_source_folder": {
        split: {str(key): int(value)
                for key, value in manifest["source_folder"].value_counts().items()}
        for split, manifest in manifest_frames.items()
    },
    "class_counts": {
        split: {str(int(key)): int(value)
                for key, value in manifest["label"].value_counts().items()}
        for split, manifest in manifest_frames.items()
    },
    "suggested_class_weights": {
        "train": {
            str(label): len(manifest_frames["train"]) /
            (max(manifest_frames["train"]["label"].nunique(), 1) * count)
            for label, count in manifest_frames["train"]["label"].value_counts().items()
        }
    },
    "train_channel_statistics_valid_pixels_only": train_stats,
    "seed": SEED, "split_fractions": SPLIT, "preprocessing": preprocessing_decisions,
}
(WORK / "dataset_info.json").write_text(json.dumps(dataset_info, indent=2, default=str))

readme = f"""# Landslide multi-source NumPy dataset

This dataset contains native-resolution Planet, Sentinel-1, and terrain inputs for a
three-branch classifier. The arrays are structured NumPy files and can be memory-mapped
without pickle.

## Files

- `train.npy`, `test.npy`, `val.npy`: records with `planet`, `s1`, `terrain`, `label`,
  `source_id`, and `sample_idx` fields.
- `test_landslide_early.npy`, `test_landslide_latest.npy`, `test_non_landslide.npy`:
  independent source-folder test sets.
- `manifest_<split>.parquet` and `manifest_test_<source>.parquet`: row-aligned metadata.
- `dataset_info.json`: shapes, channel order, preprocessing, split summaries, and train stats.
- `skipped.csv`: corrupt, missing, or incompatible samples and reasons.

## Load and use

```python
import numpy as np
arr = np.load("train.npy", mmap_mode="r")
planet, s1, terrain = arr["planet"], arr["s1"], arr["terrain"]
y, source_id = arr["label"], arr["source_id"]
test_early = np.load("test_landslide_early.npy", mmap_mode="r")
test_latest = np.load("test_landslide_latest.npy", mmap_mode="r")
test_negative = np.load("test_non_landslide.npy", mmap_mode="r")
```

Train statistics are in `dataset_info.json`; values in the arrays are raw preprocessed values
(Sentinel-1 dB and nodata zero-filled), not normalized. Use a source-specific test file to
evaluate each folder independently. To compare older versus newer events under an identical
negative pool, evaluate `landslide_early + non_landslide` and
`landslide_latest + non_landslide` separately.

Planet channels are ordered before bands then after bands. Sentinel-1 channels are ordered
before bands then after bands. Terrain channels are slope, sin(aspect), cos(aspect), and a
flat/undefined-aspect mask. See `dataset_info.json` for exact shapes and channel counts.
"""
(WORK / "README.md").write_text(readme)

for name in ("train", "test", "val"):
    (WORK / f"manifest_{name}.parquet").exists() or (_ for _ in ()).throw(
        FileNotFoundError(f"Missing manifest for {name}.")
    )
progress["finalized"] = True
save_progress()
for stage_map in stage_maps.values():
    stage_map.flush()
stage_maps.clear()
for split in SPLIT:
    (WORK / f".stage_{split}.npy").unlink(missing_ok=True)
print("Final arrays, manifests, dataset_info.json, and README.md are ready.")


## 6. Verify arrays, manifests, split isolation, and source round trips

Verification is performed before upload. A few rows are re-read from their original ZIPs
through the same preprocessing function and compared to the written memmap records. Once
arrays are finalized, restart execution at this verification cell (or at upload after a
successful verification); the preprocessing cell deliberately guards completed work.


In [ ]:
def verify_array(split, manifest):
    path = WORK / f"{split}.npy"
    arr = np.load(path, mmap_mode="r")
    print(f"{path.name}: dtype={arr.dtype}, shape={arr.shape}, size={path.stat().st_size / 1e9:.3f} GB")
    assert arr.dtype == ARRAY_DTYPE
    assert len(arr) == len(manifest)
    assert np.array_equal(arr["sample_idx"], manifest["row_index"].to_numpy(dtype=np.int32))
    for field in ("planet", "s1", "terrain"):
        assert arr[field].shape[1:] == ARRAY_DTYPE.fields[field][0].shape
        for start in range(0, len(arr), 32):
            chunk = arr[field][start:start + 32]
            assert np.isfinite(chunk).all(), f"Non-finite {field} values in {split}."
    assert np.array_equal(arr["label"], manifest["label"].to_numpy(dtype=np.int8))
    assert np.array_equal(arr["source_id"], manifest["source_id"].to_numpy(dtype=np.int8))
    assert np.array_equal(arr["source_id"], manifest["source_id"].to_numpy(dtype=np.int8))
    return arr

verified = {}
for split in SPLIT:
    verified[split] = verify_array(split, manifest_frames[split])
    print(f"{split} source counts: {Counter(manifest_frames[split]['source_folder'])}")
    print(f"{split} label counts: {Counter(manifest_frames[split]['label'])}")
for a, b in (("train", "test"), ("train", "val"), ("test", "val")):
    assert not (set(manifest_frames[a]["incident_id"]) & set(manifest_frames[b]["incident_id"]))

test_manifest = manifest_frames["test"]
for source_folder in SOURCE_LAYOUT:
    arr = np.load(WORK / f"test_{source_folder}.npy", mmap_mode="r")
    manifest = pd.read_parquet(WORK / f"manifest_test_{source_folder}.parquet")
    assert len(arr) == len(manifest)
    assert np.array_equal(arr["sample_idx"], manifest["row_index"].to_numpy(dtype=np.int32))
    assert set(manifest["source_folder"]) <= {source_folder}
    assert np.array_equal(arr["label"], manifest["label"].to_numpy(dtype=np.int8))

roundtrip_rows = test_manifest.sample(
    n=min(3, len(test_manifest)), random_state=SEED
)
for _, manifest_row in roundtrip_rows.iterrows():
    sample = next(s for s in samples if s["sample_id"] == manifest_row["sample_id"])
    zip_path = fetch_zip(sample["zip_name"])
    try:
        with tempfile.TemporaryDirectory(dir=WORK) as temp_dir, zipfile.ZipFile(zip_path) as zf:
            files = extract_sample_files(zf, sample, temp_dir)
            processed = preprocess_sample(sample, files)
            output_row = verified["test"][int(manifest_row["row_index"])]
            for field in ("planet", "s1", "terrain"):
                assert np.allclose(output_row[field], processed[field], rtol=0, atol=1e-3)
    finally:
        zip_path.unlink(missing_ok=True)
print("Round-trip checks against source TIFFs passed.")

plot_rows = test_manifest.sample(n=min(3, len(test_manifest)), random_state=SEED + 1)
fig, axes = plt.subplots(len(plot_rows), 5, figsize=(18, 4 * len(plot_rows)), squeeze=False)
for plot_i, (_, manifest_row) in enumerate(plot_rows.iterrows()):
    arr_row = verified["test"][int(manifest_row["row_index"])]
    planet = arr_row["planet"].astype(np.float32)
    band_count = planet.shape[0] // 2
    for col, image in enumerate((planet[:band_count], planet[band_count:])):
        if image.shape[0] >= 3:
            rgb = np.moveaxis(image[:3], 0, -1)
            lo, hi = np.nanpercentile(rgb, [2, 98])
            axes[plot_i, col].imshow(np.clip((rgb - lo) / max(hi - lo, 1e-6), 0, 1))
        else:
            axes[plot_i, col].imshow(image[0], cmap="gray")
        axes[plot_i, col].set_title("Planet before" if col == 0 else "Planet after")
    s1_image = arr_row["s1"].astype(np.float32)
    axes[plot_i, 2].imshow(s1_image[0], cmap="gray")
    axes[plot_i, 2].set_title("S1 before band 1 (dB)")
    axes[plot_i, 3].imshow(arr_row["terrain"][0], cmap="terrain")
    axes[plot_i, 3].set_title("Slope")
    axes[plot_i, 4].imshow(arr_row["terrain"][3], cmap="gray", vmin=0, vmax=1)
    axes[plot_i, 4].set_title("Flat/undefined aspect")
    for ax in axes[plot_i]:
        ax.axis("off")
    fig.suptitle(f"sample_id={manifest_row['sample_id']}; source={manifest_row['source_folder']}")
plt.tight_layout()
plt.show()


## 7. Evaluation helper and three-branch PyTorch Dataset example

Precision and AUC are meaningful on the combined positive/negative set, or on a landslide
folder paired with the same `non_landslide` pool. Each individual landslide folder contains
only positives and the negative folder contains only negatives.


In [ ]:
def evaluate_by_source(probs, test_arr, threshold=0.5):
    """Report source-specific detection/specificity and combined binary metrics."""
    if not 0 <= threshold <= 1:
        raise ValueError("threshold must be in [0, 1].")
    probabilities = np.asarray(probs, dtype=float).reshape(-1)
    if len(probabilities) != len(test_arr):
        raise ValueError("probs length must match test_arr.")
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("probs must be finite probabilities in [0, 1].")
    truth = np.asarray(test_arr["label"], dtype=np.int8)
    source_ids = np.asarray(test_arr["source_id"], dtype=np.int8)
    predicted = probabilities >= threshold
    result = {"by_source": {}, "overall": {}}
    source_names = {0: "landslide_early", 1: "landslide_latest", 2: "non_landslide"}
    for source_id, source_name in source_names.items():
        mask = source_ids == source_id
        if not mask.any():
            result["by_source"][source_name] = {"n": 0, "positive_rate": None}
            continue
        record = {"n": int(mask.sum()), "positive_rate": float(truth[mask].mean())}
        if source_id in (0, 1):
            record["recall_detection_rate"] = float(np.mean(predicted[mask] == 1))
            record["mean_predicted_probability"] = float(probabilities[mask].mean())
        else:
            record["specificity"] = float(np.mean(predicted[mask] == 0))
            record["false_positive_rate"] = float(np.mean(predicted[mask] == 1))
        result["by_source"][source_name] = record
    if len(np.unique(truth)) < 2:
        raise ValueError("Combined test set needs both labels for overall AUC/precision metrics.")
    result["overall"] = {
        "n": int(len(truth)), "precision": float(precision_score(truth, predicted, zero_division=0)),
        "recall": float(recall_score(truth, predicted, zero_division=0)),
        "f1": float(f1_score(truth, predicted, zero_division=0)),
        "pr_auc": float(average_precision_score(truth, probabilities)),
        "roc_auc": float(roc_auc_score(truth, probabilities)),
        "confusion_matrix": confusion_matrix(truth, predicted, labels=[0, 1]).tolist(),
    }
    return result

def evaluate_source_vs_shared_negatives(probs, test_arr, threshold=0.5):
    """Compare each positive folder against the exact same non_landslide test pool."""
    if not 0 <= threshold <= 1:
        raise ValueError("threshold must be in [0, 1].")
    probabilities = np.asarray(probs, dtype=float).reshape(-1)
    if len(probabilities) != len(test_arr):
        raise ValueError("probs length must match test_arr.")
    source_ids = np.asarray(test_arr["source_id"], dtype=np.int8)
    reports = {}
    for positive_id, name in ((0, "early_plus_non_landslide"),
                              (1, "latest_plus_non_landslide")):
        mask = np.isin(source_ids, [positive_id, 2])
        reports[name] = evaluate_by_source(probabilities[mask], test_arr[mask], threshold)
    return reports

class ThreeBranchDataset:
    """Minimal torch Dataset backed by a read-only structured NumPy memmap."""
    def __init__(self, path, train_statistics):
        import torch
        self.torch = torch
        self.arr = np.load(path, mmap_mode="r")
        self.stats = train_statistics

    def __len__(self):
        return len(self.arr)

    def __getitem__(self, index):
        torch = self.torch
        row = self.arr[index]
        result = {}
        for key in ("planet", "s1", "terrain"):
            values = np.array(row[key], dtype=np.float32, copy=True)
            mean = np.asarray(self.stats[key]["mean"], dtype=np.float32)[:, None, None]
            std = np.asarray(self.stats[key]["std"], dtype=np.float32)[:, None, None]
            result[key] = torch.from_numpy((values - mean) / std)
        result["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        result["source_id"] = torch.tensor(int(row["source_id"]), dtype=torch.long)
        return result

# Example:
# stats = json.loads(Path("dataset_info.json").read_text())["train_channel_statistics_valid_pixels_only"]
# train_dataset = ThreeBranchDataset("train.npy", stats)
# planet_batch, s1_batch, terrain_batch = (
#     batch["planet"], batch["s1"], batch["terrain"]
# )


## 8. Upload verified artifacts

The destination dataset is created as **private by default**. Upload retries are explicit;
the token is never included in logs or notebook output.


In [ ]:
api.create_repo(repo_id=DEST_REPO, repo_type="dataset", private=True, exist_ok=True)

upload_names = [
    "train.npy", "test.npy", "val.npy",
    "test_landslide_early.npy", "test_landslide_latest.npy", "test_non_landslide.npy",
    "manifest_train.parquet", "manifest_test.parquet", "manifest_val.parquet",
    "manifest_test_landslide_early.parquet", "manifest_test_landslide_latest.parquet",
    "manifest_test_non_landslide.parquet", "dataset_info.json", "README.md", "skipped.csv",
]
missing = [name for name in upload_names if not (WORK / name).is_file()]
if missing:
    raise FileNotFoundError(f"Refusing upload; expected artifacts are missing: {missing}")

def upload_with_retries(local_path, path_in_repo, attempts=5):
    """Upload one artifact with bounded exponential retry and visible failures."""
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            api.upload_file(
                path_or_fileobj=str(local_path), path_in_repo=path_in_repo,
                repo_id=DEST_REPO, repo_type="dataset",
            )
            return
        except Exception as exc:
            last_error = exc
            if attempt == attempts:
                break
            delay = min(2 ** (attempt - 1), 30)
            LOGGER.warning("Upload failed for %s (attempt %d/%d): %s; retrying in %ds",
                           path_in_repo, attempt, attempts, type(exc).__name__, delay)
            time.sleep(delay)
    raise RuntimeError(f"Upload failed after {attempts} attempts: {path_in_repo}") from last_error

for name in tqdm(upload_names, desc="Uploading artifacts"):
    upload_with_retries(WORK / name, name)

dataset_url = f"https://huggingface.co/datasets/{DEST_REPO}"
print(f"Dataset URL: {dataset_url}")
print("\nDownload snippets:")
for name in upload_names:
    print(
        f'from kaggle_secrets import UserSecretsClient\n'
        f'from huggingface_hub import hf_hub_download\n'
        f'token = UserSecretsClient().get_secret("HF_TOKEN")\n'
        f'path = hf_hub_download(repo_id="{DEST_REPO}", filename="{name}", '
        f'repo_type="dataset", token=token)'
    )
